# ReturnLoop: fine-tune a local vision model (QLoRA)
Qwen3-VL-2B, 4-bit, LoRA adapters, Unsloth, on Colab's free T4.
Same 200-photo test set and same `score()` as the Gemini run (`vision_eval.py`), so the numbers compare directly.
**Runtime -> Change runtime type -> T4 GPU**, then Run all.

In [ ]:
# setup: the repo + Unsloth (install lines from Unsloth's own Qwen3-VL notebook)
!git clone -q https://github.com/neomotion/ReturnLoopAgent
%cd ReturnLoopAgent
!pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!pip install -q transformers==4.57.6
!pip install -q --no-deps trl==0.22.2

In [ ]:
# data: 590 train / 200 test real photos (MVTec AD), 448px
%run vision_data.py   # %run, not !python: an error here stops Run all

In [ ]:
import json, time
from PIL import Image
from unsloth import FastVisionModel
from vision_data import OUT, PROMPT, answer
from vision_eval import parse, score

train = [json.loads(l) for l in (OUT / "train.jsonl").read_text().splitlines()]
test = [json.loads(l) for l in (OUT / "test.jsonl").read_text().splitlines()]

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-2B-Instruct-unsloth-bnb-4bit", load_in_4bit=True, use_gradient_checkpointing="unsloth")

def predict(row):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT.format(product=row["product"])}]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(Image.open(OUT / row["image"]), text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def evaluate(name):
    FastVisionModel.for_inference(model)
    start = time.time()
    replies = [predict(r) for r in test]
    result = score(test, [parse(t) for t in replies]) | {"seconds_per_photo": round((time.time() - start) / len(test), 2)}
    (OUT / f"result_{name}.json").write_text(json.dumps(result, indent=1))
    print(name, result, "\nexample replies:", replies[:3])
    return result

In [ ]:
# baseline: the model as downloaded, no fine-tuning
base = evaluate("local_base")

In [ ]:
# QLoRA: 4-bit frozen base + small trainable LoRA adapters on vision and language layers
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

model = FastVisionModel.get_peft_model(model, finetune_vision_layers=True, finetune_language_layers=True,
                                       r=16, lora_alpha=16, lora_dropout=0, random_state=3407)

def conversation(row):
    return {"messages": [
        {"role": "user", "content": [{"type": "text", "text": PROMPT.format(product=row["product"])},
                                     {"type": "image", "image": Image.open(OUT / row["image"])}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer(row)}]}]}

FastVisionModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=[conversation(r) for r in train],
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=2,
                   warmup_steps=5, learning_rate=2e-4, logging_steps=5, optim="adamw_8bit", seed=3407,
                   output_dir="outputs", report_to="none", remove_unused_columns=False,
                   dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True}, max_length=2048))
stats = trainer.train()
print(f"trained in {stats.metrics['train_runtime'] / 60:.1f} min")

In [ ]:
# fine-tuned, same test set
tuned = evaluate("local_tuned")
print("\n            base  ->  tuned")
for k in base:
    print(f"{k:18} {base[k]}  ->  {tuned[k]}")

In [ ]:
# keep the adapter (~70 MB) and the results in your Google Drive
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/ReturnLoopAgent
model.save_pretrained("/content/drive/MyDrive/ReturnLoopAgent/qwen3vl_2b_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/ReturnLoopAgent/qwen3vl_2b_lora")
!cp data/vision/result_local_*.json /content/drive/MyDrive/ReturnLoopAgent/